In [2]:
import torch
import torch.nn as nn
import math
import pandas as pd
import matplotlib.pyplot as plt

from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [3]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

baseline_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
baseline_model.eval()

print("Baseline GPT-2 loaded.")

Baseline GPT-2 loaded.


In [4]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_dataset.set_format(type="torch")

train_loader = DataLoader(tokenized_dataset["train"], batch_size=8, shuffle=True)
eval_loader = DataLoader(tokenized_dataset["validation"], batch_size=8)

print("Dataset ready.")

Dataset ready.


In [5]:
def evaluate_perplexity(model, dataloader):
    model.eval()
    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            labels = input_ids.clone()
            labels[attention_mask == 0] = -100

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            total_loss += loss.item() * input_ids.size(0)
            total_tokens += input_ids.size(0)

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)

In [6]:
baseline_ppl = evaluate_perplexity(baseline_model, eval_loader)
print("Baseline GPT-2 perplexity:", baseline_ppl)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Baseline GPT-2 perplexity: 69.17921938374056


In [7]:
def kronecker_decompose_rank_k(W, m1, m2, n1, n2, k=2):
    """
    W shape: (m1*m2, n1*n2)
    """
    W_reshaped = W.reshape(m1, m2, n1, n2)
    W_reshaped = W_reshaped.permute(0, 2, 1, 3)
    W_reshaped = W_reshaped.reshape(m1*n1, m2*n2)

    U, S, Vh = torch.linalg.svd(W_reshaped)

    A_list = []
    B_list = []

    for i in range(k):
        u = U[:, i].reshape(m1, n1)
        v = Vh[i, :].reshape(m2, n2)

        A = torch.sqrt(S[i]) * u
        B = torch.sqrt(S[i]) * v

        A_list.append(A)
        B_list.append(B)

    return A_list, B_list

In [8]:
class KroneckerLinearRankK(nn.Module):
    def __init__(self, A_list, B_list, bias):
        super().__init__()
        
        self.A_list = nn.ParameterList([nn.Parameter(A) for A in A_list])
        self.B_list = nn.ParameterList([nn.Parameter(B) for B in B_list])
        self.bias = nn.Parameter(bias.clone()) if bias is not None else None

    def forward(self, x):
        W_total = 0
        
        for A, B in zip(self.A_list, self.B_list):
            W_total = W_total + torch.kron(A, B)

        out = torch.matmul(x, W_total.t())

        if self.bias is not None:
            out = out + self.bias

        return out

In [9]:
def compress_one_layer_rank2(model, layer_idx):
    block = model.transformer.h[layer_idx]

    # ======================
    # c_fc  (3072 x 768)
    # ======================
    W_fc = block.mlp.c_fc.weight
    b_fc = block.mlp.c_fc.bias

    # 3072 = 48 × 64
    # 768  = 12 × 64
    A_fc, B_fc = kronecker_decompose_rank_k(
        W_fc,
        m1=48, m2=64,
        n1=12, n2=64,
        k=2
    )

    block.mlp.c_fc = KroneckerLinearRankK(A_fc, B_fc, b_fc)

    # ======================
    # c_proj (768 x 3072)
    # ======================
    W_proj = block.mlp.c_proj.weight
    b_proj = block.mlp.c_proj.bias

    # 768  = 12 × 64
    # 3072 = 48 × 64
    A_proj, B_proj = kronecker_decompose_rank_k(
        W_proj.T,
        m1=12, m2=64,
        n1=48, n2=64,
        k=2
    )

    block.mlp.c_proj = KroneckerLinearRankK(A_proj, B_proj, b_proj)

    return model

In [10]:
rank2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
rank2_model.eval()

for layer_idx in range(12):
    compress_one_layer_rank2(rank2_model, layer_idx)

print("All 12 layers compressed with Rank-2.")

All 12 layers compressed with Rank-2.


In [11]:
ppl_rank2_before = evaluate_perplexity(rank2_model, eval_loader)
print("Rank-2 12-layer perplexity (before training):", ppl_rank2_before)

Rank-2 12-layer perplexity (before training): 21978.30694074659


In [ ]:
optimizer = AdamW(rank2_model.parameters(), lr=3e-5)
scaler = GradScaler(device="cuda")

rank2_model.train()

for epoch in range(5):
    total_loss = 0
    steps = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        optimizer.zero_grad()

        with autocast(device_type="cuda"):
            outputs = rank2_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        steps += 1

    print(f"Epoch {epoch+1}, Avg Loss: {total_loss/steps}")

Epoch 1, Avg Loss: 6.194455523844119


In [ ]:
ppl_rank2_after = evaluate_perplexity(rank2_model, eval_loader)
print("Rank-2 12-layer perplexity (after training):", ppl_rank2_after)

In [ ]:
results_rank2 = pd.DataFrame({
    "Model": [
        "Original GPT-2",
        "Rank-2 12-layer compressed"
    ],
    "Perplexity": [
        baseline_ppl,
        ppl_rank2_after
    ]
})

results_rank2

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

baseline_params = count_parameters(baseline_model)
rank2_params = count_parameters(rank2_model)

print("Baseline parameters:", f"{baseline_params:,}")
print("Rank-2 parameters:", f"{rank2_params:,}")

reduction = (baseline_params - rank2_params) / baseline_params * 100
print(f"Parameter reduction: {reduction:.2f}%")

In [1]:
import torch
import time
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

baseline_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
baseline_model.eval()

print("Baseline model loaded.")

Baseline model loaded.


In [3]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

baseline_params = count_parameters(baseline_model)
rank2_params = count_parameters(rank2_model)

print("Baseline parameters:", f"{baseline_params:,}")
print("Rank-2 parameters:", f"{rank2_params:,}")

reduction = (baseline_params - rank2_params) / baseline_params * 100
print(f"Parameter reduction: {reduction:.2f}%")

NameError: name 'rank2_model' is not defined